In [ ]:
# ==============================================================================
# STEP 1: MASTER CONSOLIDATION, COMPREHENSIVE QA, & CDC HASHING
# 
# ARCHITECTURE NOTE: This script evaluates all raw files and aggregates QA 
# errors. If successful, it checks for dropped CURIEs against the prior master
# dataset (saving an audit log), generates a cryptographic hash of semantic 
# columns for Change Data Capture (CDC), and overwrites the Silver Layer master.
# ==============================================================================

import os
import sys
import glob
import hashlib
import datetime
import pandas as pd
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))
error_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim", "logs"))

os.makedirs(interim_data_dir, exist_ok=True)
os.makedirs(error_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)

try:
    from ingest_schema_manager import COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Ensure config directory is accessible.")

master_output_file = os.path.join(interim_data_dir, "master_ontology_dataset.csv")

# --- 2. Ingestion & Individual File QA ---
print("[*] Initiating Step 1 Comprehensive Consolidation...")
raw_files = glob.glob(os.path.join(raw_data_dir, "raw_*.csv"))

if not raw_files:
    raise FileNotFoundError(f"CRITICAL: No raw CSV files found in {raw_data_dir}.")

df_list = []
error_log = []
total_raw_rows = 0

for file_path in raw_files:
    filename = os.path.basename(file_path)
    print(f"    [+] Reading {filename}...")
    
    try:
        df = pd.read_csv(file_path, dtype={'Concept_ID': str, 'CURIE': str, 'Parent_IDs': str})
    except Exception as e:
        error_log.append(f"[File Read Error] {filename}: {e}")
        continue
    
    # QA GATE 1: Schema Compliance
    if list(df.columns) != COLUMN_ORDER:
        error_log.append(
            f"[Schema Mismatch] {filename} has {len(df.columns)} columns instead of {len(COLUMN_ORDER)}. "
            f"Missing/Extra: {set(df.columns).symmetric_difference(set(COLUMN_ORDER))}"
        )
        continue 
        
    # QA GATE 2: Null Primary Keys
    null_curies = df['CURIE'].isnull().sum()
    if null_curies > 0:
        error_log.append(f"[Null Primary Keys] {filename} contains {null_curies} rows with missing CURIEs.")
        continue 
        
    row_count = len(df)
    total_raw_rows += row_count
    df_list.append(df)

# --- 3. Global QA & Deduplication Checks ---
if df_list:
    print("\n[*] Merging valid datasets for Global QA checks...")
    master_df = pd.concat(df_list, ignore_index=True)

    # QA GATE 3: Exact Row Duplication
    exact_dupes = master_df[master_df.duplicated(keep=False)]
    if not exact_dupes.empty:
        error_file = os.path.join(error_dir, "error_exact_duplicates.csv")
        exact_dupes.sort_values('CURIE').to_csv(error_file, index=False, encoding='utf-8-sig')
        error_log.append(f"[Exact Duplicates] {len(exact_dupes)} completely identical rows found. Exported to {os.path.basename(error_file)}")

    # QA GATE 4: CURIE Collisions (Different data, same ID)
    temp_deduped = master_df.drop_duplicates()
    curie_collisions = temp_deduped[temp_deduped.duplicated(subset=['CURIE'], keep=False)]
    
    if not curie_collisions.empty:
        error_file = os.path.join(error_dir, "error_curie_collisions.csv")
        curie_collisions.sort_values('CURIE').to_csv(error_file, index=False, encoding='utf-8-sig')
        error_log.append(f"[CURIE Collisions] {len(curie_collisions)} conflicting rows share the same CURIE. Exported to {os.path.basename(error_file)}")

if error_log:
    print("\n" + "="*60)
    print(" CRITICAL CONSOLIDATION FAILURES DETECTED ")
    print("="*60)
    for error in error_log:
        print(f" - {error}")
    print("="*60)
    print("\n[!] The master dataset was NOT generated. Please resolve the errors above in your raw files.")
    sys.exit(1)
    
if not df_list:
    print("\n[!] No valid data passed the file-level QA gates. Exiting.")
    sys.exit(1)

master_df = master_df.drop_duplicates() 

# --- 4. Dropped CURIE Check & Logging ---
# Compares the newly consolidated data against the existing master file to detect deletions
if os.path.exists(master_output_file):
    print("\n[*] Auditing against previous master dataset for dropped records...")
    try:
        old_master_df = pd.read_csv(master_output_file, dtype={'CURIE': str})
        old_curies = set(old_master_df['CURIE'].dropna())
        new_curies = set(master_df['CURIE'].dropna())
        
        dropped_curies = old_curies - new_curies
        
        if dropped_curies:
            dropped_df = old_master_df[old_master_df['CURIE'].isin(dropped_curies)][['CURIE', 'Primary_Label', 'Source_System', 'Hierarchy_Path']]
            
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            drop_log_path = os.path.join(error_dir, f"dropped_curies_log_{timestamp}.csv")
            dropped_df.to_csv(drop_log_path, index=False, encoding='utf-8-sig')
            
            print("\n" + "!"*60)
            print(f" WARNING: {len(dropped_curies)} CURIEs DISCOVERED MISSING FROM RAW EXTRACTS")
            print("!"*60)
            print(f"[*] Full deletion log saved to: {drop_log_path}")
            for _, row in dropped_df.head(5).iterrows():
                print(f"  - {row['CURIE']} | {row['Primary_Label']} ({row['Source_System']})")
            if len(dropped_curies) > 5:
                print(f"  ... and {len(dropped_curies) - 5} more.")
            print("!"*60 + "\n")
        else:
            print("[+] No records were dropped from the previous extraction.")
    except Exception as e:
        print(f"[!] Warning: Could not execute dropped CURIE check: {e}")

# --- 5. Change Data Capture (CDC) Hashing ---
def generate_row_hash(row):
    """Generates a deterministic SHA-256 hash of core semantic columns to detect metadata changes."""
    core_str = f"{row.get('Primary_Label', '')}|{row.get('Description', '')}|{row.get('Hierarchy_Path', '')}|{row.get('Parent_IDs', '')}|{row.get('Synonyms', '')}"
    return hashlib.sha256(core_str.encode('utf-8')).hexdigest()

print("[*] Generating SHA-256 semantic hashes for Change Data Capture...")
master_df['row_hash'] = master_df.apply(generate_row_hash, axis=1)

# Ensure strict column ordering, appending the new hash column at the end
final_columns = COLUMN_ORDER + ['row_hash']
master_df = master_df[final_columns]

# --- 6. Final Export ---
print(f"[*] Writing Master Dataset to {os.path.basename(master_output_file)}...")
master_df.to_csv(master_output_file, index=False, encoding='utf-8-sig')
print(f"[COMPLETE] Consolidation finished. Final dataset contains {len(master_df)} distinct concepts.")